# **Building Your First Tool**

**Written by**:
- Tony Menzo (U. of Alabama / Fermilab)

A beginner's guide to writing tools that an LLM agent can call, and to
packaging them so [toolbase](https://github.com/alexr314/toolbase) can serve them
to Claude Code, Codex, opencode, or any other MCP-capable agent.

**No prior experience with agents is assumed.** If you can write a Python
function, you can write a tool.

**What you'll learn:**

1. What a "tool" actually is, from the agent's point of view
2. **Template A** — turn a plain function into a tool with `@define_tool`
3. **Template B** — write a tool as a class by subclassing `BaseTool`
4. A realistic workflow: **pull events from a database** $\to$ **feed them to
   external analysis code** $\to$ **get observables back**
5. How to package your tools as a **toolbase toolkit** so an agent can use them

**The physics example:** a toy dimuon analysis. We pull muon-pair events out of a
database, hand them to an external physics package, and compute the dimuon
invariant-mass spectrum  (inspired by workflows used in a real CMS $Z \to \mu^+\mu^-$
measurement, just with far less machinery in the way).

---

## 0. Prerequisites

### Environment

```bash
conda create -n heptapod python=3.13
conda activate heptapod
pip install -r requirements.txt     # from the repository root
```

The only hard requirements for this notebook are `orchestral-ai` and
`matplotlib`. Everything else (the "database", the "external package") is
created by the notebook itself in a local `sandbox/` directory.

### API key (only for Section 4.6, feel free to skip)

Sections 1–5 run with **no API key at all** — you can build and test tools
entirely offline. Only the one cell where we hand the tools to a live agent
needs a key. Put it in a `.env` file at the repository root:

```bash
OPENAI_API_KEY=sk-...           # or ANTHROPIC_API_KEY, GOOGLE_API_KEY, GROQ_API_KEY
```

### toolbase (only for Section 5)

```bash
pip install toolbase            # provides the `tb` command
```

In [ ]:
# ============================================================================
# SETUP - run this cell first
# ============================================================================
import sys, os, json, math, random, sqlite3, subprocess
from pathlib import Path

# Add the repository root to the path (examples/primer/ -> 2 levels up)
REPO_ROOT = Path.cwd().parent.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# Everything this notebook writes goes in here, so it is easy to delete later
SANDBOX = Path.cwd() / "sandbox"
(SANDBOX / "external_packages").mkdir(parents=True, exist_ok=True)
base_directory = str(SANDBOX)

# Load API keys, if you have a .env (only needed in Section 4.6)
try:
    from dotenv import load_dotenv
    load_dotenv(REPO_ROOT / ".env")
except ImportError:
    pass

print(f"Repository root : {REPO_ROOT}")
print(f"Working directory: {base_directory}")

---

## 1. What is a tool?

An LLM cannot run MadGraph, query a database, or read a file. It can only emit
text. A **tool** is the bridge: a Python function that you expose to the model,
so that instead of *describing* what it wants, the model can *ask for it*.

From the agent's side, a tool is exactly three things:

| Ingredient | Where it comes from | Why it matters |
|---|---|---|
| **A name** | the function or class name | how the model refers to it |
| **A description** | the **docstring** | this is the *only* documentation the model gets |
| **Typed arguments** | the function signature / fields | tells the model what to fill in |

and one rule for what comes back:

> **A tool always returns a string.** Usually a short JSON string. The return
> value is read by a language model, not by your Python code, so it must be
> readable text.

The single most common mistake is writing a thin docstring. The
docstring *is* the user manual, and its reader is the model. Write it for a
capable colleague who has never seen your code.

There are two ways to write a tool. Both are shown below; they produce the same
kind of object.

| | **Template A** — `@define_tool` | **Template B** — `BaseTool` subclass |
|---|---|---|
| Looks like | a function | a class |
| Good for | pure calculations, quick wrappers | anything with configuration, files, or setup |
| Configuration | awkward | first-class (`StateField`) |
| Used by HEPTAPOD | occasionally | **for every shipped tool** |

Start with A. Move to B the moment your tool needs to know a path, a database
location, or an API key.

---

## 2. Template A — `@define_tool`

Write a normal Python function. Add type hints. Write a real docstring. Put
`@define_tool()` on top. That is the whole recipe.

In [ ]:
from orchestral.tools import define_tool


@define_tool(param_descriptions={
    "px": "Momentum along the x axis, in GeV",
    "py": "Momentum along the y axis, in GeV",
})
def transverse_momentum(px: float, py: float) -> str:
    """
    Compute the transverse momentum pT = sqrt(px^2 + py^2) of a particle.

    Use this when you have the Cartesian momentum components of a particle and
    need its momentum in the plane transverse to the beam.

    Args:
        px: Momentum along the x axis, in GeV.
        py: Momentum along the y axis, in GeV.

    Returns:
        A one-line string with the transverse momentum in GeV.
    """
    pt = math.sqrt(px**2 + py**2)
    return f"pT = {pt:.3f} GeV"


print(type(transverse_momentum))

Note what `@define_tool()` handed back: **not a function, but a ready-to-use
tool object**. You no longer call it like `transverse_momentum(3.0, 4.0)`. You
call `.execute()` with keyword arguments — the same entry point the agent uses,
including its input validation.

In [ ]:
# Run the tool by hand, exactly the way the agent will run it
print(transverse_momentum.execute(px=3.0, py=4.0))

### 2.1 What the model actually sees

Before handing a tool to an agent, it is worth looking at the description the
agent receives. If it reads badly to you, it reads badly to the model.

In [ ]:
print("Tool name the model sees:", transverse_momentum.get_name())
print()
print("Argument schema the model sees:")
print(json.dumps(transverse_momentum.get_input_schema(), indent=2))

The `param_descriptions` argument is what filled in those `"description"`
fields. Leave it out and each argument is described by its own name — `"px":
"px"` — which tells the model nothing. **Always pass `param_descriptions`.**

### 2.2 Rules of thumb

- **One tool, one job.** `fetch_events` and `compute_mass` are two tools, not
  one tool with a `mode` argument.
- **Return a string.** Return a JSON string when there is structure to report,
  a plain sentence when there is not.
- **Never raise.** An exception is invisible to the model. Catch the problem and
  return a message that says what went wrong *and how to fix it*.
- **Name arguments in full.** `min_pt`, not `p`. The name is documentation.

---

## 3. Template B — subclassing `BaseTool`

Every tool that ships with HEPTAPOD is written this way. It is slightly more
typing than a decorated function, and it buys you three things: configuration
that the model never sees, a `_setup()` hook that runs once, and structured
errors.

The anatomy:

```python
class MyTool(BaseTool):
    """Docstring -> what the model reads."""

    thing: float = RuntimeField(description="...")   # the model fills this in
    setting: str  = StateField(description="...")    # YOU fill this in

    def _setup(self):        # optional: runs once, at construction
        ...

    def _run(self) -> str:   # required: the actual work
        return "..."
```

In [ ]:
from orchestral.tools import BaseTool
from orchestral.tools.base.field_utils import RuntimeField, StateField


class RapidityTool(BaseTool):
    """
    Compute the rapidity y = 0.5 * ln((E + pz) / (E - pz)) of a particle.

    Use this when you have a particle's energy and its momentum along the beam
    axis, and you need the rapidity. Requires E > |pz|.

    Args:
        energy: Particle energy, in GeV.
        pz: Momentum along the beam (z) axis, in GeV.

    Returns:
        A one-line string with the rapidity.
    """

    # ==================== Runtime fields (the model provides these) ==========
    energy: float = RuntimeField(description="Particle energy in GeV")
    pz: float = RuntimeField(description="Momentum along the beam (z) axis in GeV")
    # =========================================================================

    # ==================== State fields (you provide these) ===================
    units: str = StateField(default="GeV", description="Units used for the inputs")
    # =========================================================================

    def _run(self) -> str:
        if self.energy <= abs(self.pz):
            return self.format_error(
                error="Unphysical Input",
                reason="energy must be larger than |pz|",
                context=f"energy={self.energy}, pz={self.pz}",
                suggestion="Check that energy and pz come from the same four-vector",
            )
        y = 0.5 * math.log((self.energy + self.pz) / (self.energy - self.pz))
        return f"rapidity y = {y:.4f}  (inputs in {self.units})"


rapidity_tool = RapidityTool()          # state fields are set here, once
print("Tool name:", rapidity_tool.get_name())

In [ ]:
# The good case
print(rapidity_tool.execute(energy=100.0, pz=60.0))
print()
# The bad case: no traceback, just a message the model can act on
print(rapidity_tool.execute(energy=10.0, pz=60.0))

### 3.1 `RuntimeField` vs. `StateField`

This is the one distinction worth getting right, and it is not about types — it
is about **who is allowed to decide the value**.

| | `RuntimeField` | `StateField` |
|---|---|---|
| Who sets it | the **model**, on every call | **you**, once, at construction |
| Visible to the model | yes | **no** |
| Typical contents | dataset name, cut value, file to read | file paths, database location, API keys |
| Set by | the agent's tool call | your Python code, or toolbase config |

Rule of thumb: **if the model guessing the value would be bad, it is a
`StateField`.** A database path is a `StateField`. Which dataset to read is a
`RuntimeField`.

This is also a safety boundary. The model can ask for `dataset="DoubleMuon_RunA"`;
it cannot ask for a different database.

### 3.2 Errors: `format_error()`

Never let an exception escape `_run()`. Use `self.format_error()`, which
produces a message with a shape the model has learned to read:

```python
return self.format_error(
    error="File Not Found",              # short category
    reason="the event file does not exist",   # what happened
    context=f"events_file={self.events_file}",# the offending value
    suggestion="Run FetchEvents first to create it",   # what to do about it
)
```

The `suggestion` field is what turns a dead end into a recovery. An agent that
reads *"Run FetchEvents first"* will do exactly that, and your workflow
self-corrects instead of stopping.

---

## 4. A realistic workflow: database $\to$ events $\to$ observables

Most analysis workflows have the same skeleton, whatever the experiment and
whatever the data:

1. **Query a database** for the events you want.
2. **Write those events out** — they are now a dataset on disk.
3. **Compute observables** from them, using analysis code that already exists.

Very little of that is specific to the physics. What differs between groups is
which database, which file format, and whose library computes the observables, those
are details that live inside one tool, not a complete change to the shape of the
workflow.

This naturally breaks down into two tools: **`FetchEvents`** (database $\to$ events)
and **`ComputeObservables`** (events $\to$ observables).

You might ask, *why not one tool*? A few good reasons:

- The agent can **re-run step 3 with different settings** without re-querying the
  database.
- If step 3 fails, the events are still on disk — nothing is lost.
- Each tool has a docstring describing **one** thing, so the model picks
  correctly.

Below we consider a toy example similar in shape to a (type of) CMS columnar analysis:

| This notebook | A typical CMS analysis |
|---|---|
| query the SQLite database | locate the dataset (Rucio), open its NanoAOD files with `uproot`, usually over `xrootd` |
| events written to a JSON file, one array per field | events as `awkward` arrays, wrapped by `coffea`'s `NanoEvents` |
| `dimuon_lib.compute_observables()` | a `coffea` processor: selections and observables |
| the histogram at the end | `hist` objects the processor fills |

Two differences to point out with the real thing:

- Like NanoAOD, we'll consider the events file to hold one
  array per field rather than a list of per-event records. Unlike NanoAOD, every
  event here has exactly two muons, so those arrays are rectangular and plain
  `numpy` is enough. Real data has a variable number of muons per event calling for an `awkward` array. Exercise 6 puts it back.
- Typically `coffea` feeds NanoAOD straight into the
  processor without writing an intermediate events file. Writing one here is
  deliberate as it allows the agent to retry the second step without
  re-querying. (similar to the purpose of a skim)

### 4.1 Build the toy event database

Here we fabricate muon pairs whose invariant mass is drawn either from a $Z$ resonance at $91.19$ GeV or from a falling continuum, with a couple of percent of momentum smearing to imitate detector resolution.

The table is called `Events` and its columns follow **NanoAOD** (`run`, `luminosityBlock`, `event`, `nMuon`, `Muon_pt`, ...) so the names you handle here are the names you will meet in real data.

**Nothing in this cell is agent-related.** It is just making a database exist.

In [ ]:
DB_PATH = SANDBOX / "cms_dimuon.db"

random.seed(1234)
M_Z, GAMMA_Z = 91.19, 2.50          # GeV


def _draw_mass():
    """Draw a dimuon mass: 75% Z resonance, 25% falling continuum."""
    if random.random() < 0.75:
        m = M_Z + 0.5 * GAMMA_Z * math.tan(math.pi * (random.random() - 0.5))
        return m if 60.0 < m < 120.0 else M_Z
    return 20.0 * math.exp(random.random() * 1.1)


def _make_event(event_number, dataset):
    """Build one back-to-back muon pair with the target invariant mass."""
    m = _draw_mass()
    eta = random.uniform(-1.5, 1.5)
    phi = random.uniform(-math.pi, math.pi)
    pt = m / (2.0 * math.cosh(eta))                  # exact for a back-to-back pair
    smear = lambda x: x * (1.0 + random.gauss(0.0, 0.02))   # 2% momentum resolution
    return (dataset, 195000 + event_number % 7, 1 + event_number % 50,
            event_number, 2,
            json.dumps([smear(pt), smear(pt)]),      # Muon_pt   : one list per event
            json.dumps([eta, -eta]),                 # Muon_eta
            json.dumps([phi, phi + math.pi]),        # Muon_phi
            json.dumps([1, -1]))                     # Muon_charge


DB_PATH.unlink(missing_ok=True)
con = sqlite3.connect(DB_PATH)

# NanoAOD keeps one branch per field, and the Muon_* branches are "jagged":
# each event carries a list, because the number of muons varies event to event.
# Storing those lists as JSON text is the simplest stand-in for that shape.
con.execute("""
    CREATE TABLE Events (
        dataset   TEXT,    run      INTEGER, luminosityBlock INTEGER,
        event     INTEGER, nMuon    INTEGER,
        Muon_pt   TEXT,    Muon_eta TEXT,    Muon_phi TEXT,  Muon_charge TEXT)
""")
rows  = [_make_event(i, "DoubleMuon_RunA") for i in range(2000)]
rows += [_make_event(i, "DoubleMuon_RunB") for i in range(2000, 2800)]
con.executemany("INSERT INTO Events VALUES (?,?,?,?,?,?,?,?,?)", rows)
con.commit()

print(f"Database written to {DB_PATH.name}")
for dataset, n in con.execute(
        "SELECT dataset, COUNT(*) FROM Events GROUP BY dataset"):
    print(f"  {dataset:20s} {n:5d} events")
con.close()

In [ ]:
# One row of the database. Each Muon_* field holds a list -- one entry per muon.
con = sqlite3.connect(DB_PATH)
con.row_factory = sqlite3.Row
row = con.execute("SELECT * FROM Events LIMIT 1").fetchone()
con.close()

for key in row.keys():
    print(f"  {key:16s} {row[key]}")

### 4.2 The external physics package

Now we need the code that actually does physics. Pretend
a collaborator wrote `dimuon_lib` and you installed it with `pip`. It takes
columns of event data and returns columns of observables (similar in spirit to the way a
`coffea` processor works). It has **never heard of an LLM**.

This separation is a key point:

> **Physics goes in a library. The tool is a thin adapter around it.**

Keep it that way and your physics stays testable with `pytest`, reviewable by
your collaboration, and reusable outside any agent. The tool layer just
translates between "arguments from a model" and "a function call".

In [ ]:
%%writefile sandbox/external_packages/dimuon_lib.py
"""
dimuon_lib -- an ordinary physics library.

Pretend a collaborator wrote this and you installed it with pip. It knows
nothing about agents, LLMs, or toolbase.

Everything here is columnar, the way NanoAOD stores events and the way a coffea
processor reads them: one array per field, indexed by event, with the two muons
of an event along the second axis. No loops over events -- numpy does them all
at once.
"""
import numpy as np


def invariant_mass(events):
    """Dimuon invariant mass, in GeV, for every event at once."""
    pt = np.asarray(events["Muon_pt"])              # shape (n_events, 2)
    eta = np.asarray(events["Muon_eta"])
    phi = np.asarray(events["Muon_phi"])
    # m^2 = 2 pt1 pt2 (cosh(deta) - cos(dphi))  for massless muons
    return np.sqrt(2 * pt[:, 0] * pt[:, 1] *
                   (np.cosh(eta[:, 0] - eta[:, 1]) - np.cos(phi[:, 0] - phi[:, 1])))


def pair_pt(events):
    """Transverse momentum, in GeV, of the muon pair, for every event at once."""
    pt = np.asarray(events["Muon_pt"])
    phi = np.asarray(events["Muon_phi"])
    px = (pt * np.cos(phi)).sum(axis=1)
    py = (pt * np.sin(phi)).sum(axis=1)
    return np.hypot(px, py)


def compute_observables(events):
    """Map columns of event data to columns of observables."""
    charge = np.asarray(events["Muon_charge"])
    return {
        "event": list(events["event"]),
        "mass": invariant_mass(events).tolist(),
        "pair_pt": pair_pt(events).tolist(),
        "opposite_sign": (charge.prod(axis=1) < 0).tolist(),
    }


def summarize(observables):
    """Summary statistics for a column of masses."""
    masses = np.asarray(observables["mass"])
    if masses.size == 0:
        return {"n_events": 0}
    return {
        "n_events": int(masses.size),
        "mean_mass": float(masses.mean()),
        "median_mass": float(np.median(masses)),
        "min_mass": float(masses.min()),
        "max_mass": float(masses.max()),
    }

In [ ]:
# Make the "installed package" importable, and check it works on its own
sys.path.insert(0, str(SANDBOX / "external_packages"))
import dimuon_lib

# A single event, in the columnar layout: one list per field, one event deep
one_event = {"event":       [0],
             "Muon_pt":     [[45.0, 45.0]],
             "Muon_eta":    [[0.0, 0.0]],
             "Muon_phi":    [[0.0, math.pi]],
             "Muon_charge": [[1, -1]]}

print(f"invariant mass = {dimuon_lib.invariant_mass(one_event)[0]:.2f} GeV")

### 4.3 Tool 1 — `FetchEventsTool`

Pull events out of the database and write them to a file.

Read the fields before the code: `dataset`, `max_events` and `output_file` are
things the **model** should choose per call. `db_path` and `base_directory` are
things **you** decide once, the model should never be able to point this tool
at a different database or write outside the sandbox.

Note `_safe_path()`. Every tool that touches the filesystem needs something like
it: the model supplies file names, and `"../../.ssh/id_rsa"` is a file name too.
Four lines, and the sandbox actually holds.

In [ ]:
# NanoAOD-style field names. The Muon_* columns hold one list per event.
SCALAR_COLUMNS = ["run", "luminosityBlock", "event", "nMuon"]
MUON_COLUMNS = ["Muon_pt", "Muon_eta", "Muon_phi", "Muon_charge"]

KNOWN_DATASETS = "DoubleMuon_RunA, DoubleMuon_RunB"


class FetchEventsTool(BaseTool):
    """
    Fetch dimuon events from the experiment's event database and write them to a
    JSON file in the working directory, one array per field.

    Call this FIRST, before computing any observables. Available datasets are
    'DoubleMuon_RunA' and 'DoubleMuon_RunB'.

    Args:
        dataset: Name of the dataset to read.
        max_events: How many events to pull (default 500).
        output_file: File name to write, relative to the working directory.

    Returns (JSON):
        {"status": "ok", "dataset": "<name>", "n_events": <int>,
         "output_file": "<name>"}

    Errors:
        Returns a formatted error if the database is missing, the dataset name
        is unknown, or output_file points outside the working directory.
    """

    # ==================== Runtime fields (the model provides these) ==========
    dataset: str = RuntimeField(
        description="Dataset name, one of 'DoubleMuon_RunA' or 'DoubleMuon_RunB'")
    max_events: int = RuntimeField(
        default=500, description="Maximum number of events to fetch")
    output_file: str = RuntimeField(
        default="events.json",
        description="Output file name, relative to the working directory")
    # =========================================================================

    # ==================== State fields (you provide these) ===================
    db_path: str = StateField(description="Path to the event database file")
    base_directory: str = StateField(description="Working directory for file output")
    # =========================================================================

    def _safe_path(self, filename: str) -> Path:
        """Resolve a file name inside base_directory, and refuse to escape it."""
        root = Path(self.base_directory).resolve()
        full = (root / filename).resolve()
        if root not in full.parents and full != root:
            raise ValueError(f"path escapes the working directory: {filename}")
        return full

    def _run(self) -> str:
        if not Path(self.db_path).exists():
            return self.format_error(
                error="Database Not Found",
                reason="the event database file does not exist",
                context=f"db_path={self.db_path}",
                suggestion="Check db_path when constructing the tool")

        try:
            out_path = self._safe_path(self.output_file)
        except ValueError as exc:
            return self.format_error(
                error="Access Denied", reason=str(exc),
                suggestion="Use a plain file name such as 'events.json'")

        con = sqlite3.connect(self.db_path)
        con.row_factory = sqlite3.Row
        rows = con.execute(
            f"SELECT {', '.join(SCALAR_COLUMNS + MUON_COLUMNS)} FROM Events "
            "WHERE dataset = ? ORDER BY event LIMIT ?",
            (self.dataset, self.max_events)).fetchall()
        con.close()

        if not rows:
            return self.format_error(
                error="No Events Found",
                reason=f"dataset '{self.dataset}' returned no rows",
                suggestion=f"Try one of: {KNOWN_DATASETS}")

        # Columnar layout: one array per field, the way NanoAOD stores events
        events = {col: [row[col] for row in rows] for col in SCALAR_COLUMNS}
        events.update({col: [json.loads(row[col]) for row in rows]
                       for col in MUON_COLUMNS})
        out_path.write_text(json.dumps(events))

        # Return a SUMMARY plus a path -- never the events themselves
        return json.dumps({
            "status": "ok",
            "dataset": self.dataset,
            "n_events": len(rows),
            "output_file": self.output_file,
        })


fetch_tool = FetchEventsTool(db_path=str(DB_PATH), base_directory=base_directory)
print("Created:", fetch_tool.get_name())

In [ ]:
# Try it by hand, before any agent is involved
print(fetch_tool.execute(dataset="DoubleMuon_RunA", max_events=500))

events = json.loads((SANDBOX / "events.json").read_text())
print(f"\nFields on disk : {list(events)}")
print(f"Events         : {len(events['event'])}")
print(f"Muon_pt[:2]    : {events['Muon_pt'][:2]}")

**Test your tools by hand first.** A tool that misbehaves under an agent is
nearly impossible to debug, because you are watching two unpredictable systems
at once. Get `.execute()` right in a notebook cell, then hand it over.

That includes the failure paths, those messages are what the agent reads when
it goes off track, so they deserve as much care as the happy path:

In [ ]:
print(fetch_tool.execute(dataset="DoubleMuon_RunC"))          # typo in the dataset name
print()
print(fetch_tool.execute(dataset="DoubleMuon_RunA",
                         output_file="../../escape.json"))     # attempted sandbox escape

### 4.4 Tool 2: `ComputeObservablesTool`

The second tool reads the file the first one wrote, hands the events to
`dimuon_lib`, and writes the observables back out.

The tool simply resolves two paths, reads a file, calls
**one** library function, writes a file, and returns a summary. No physics lives
here (which is fine). Notice too that it never touches a field name, so it did not have to know
that the events are stored columnwise at all. If the invariant-mass definition
ever changes, it changes in `dimuon_lib`, where it belongs, and the tool does
not move.

In [ ]:
class ComputeObservablesTool(BaseTool):
    """
    Compute dimuon observables (invariant mass, pair pT) for a file of events
    produced by FetchEvents, and write them to a JSON file.

    Call this AFTER FetchEvents. The events_file argument must be the
    output_file that FetchEvents reported.

    Args:
        events_file: JSON event file written by FetchEvents, e.g. 'events.json'.
        output_file: File name for the observables, relative to the working
            directory.

    Returns (JSON):
        {"status": "ok", "output_file": "<name>", "n_events": <int>,
         "mean_mass": <GeV>, "median_mass": <GeV>,
         "min_mass": <GeV>, "max_mass": <GeV>}

    Errors:
        Returns a formatted error if the event file does not exist, or if a
        file name points outside the working directory.
    """

    # ==================== Runtime fields (the model provides these) ==========
    events_file: str = RuntimeField(
        description="Event file to read, as reported by FetchEvents, e.g. 'events.json'")
    output_file: str = RuntimeField(
        default="observables.json",
        description="Output file name for the computed observables")
    # =========================================================================

    # ==================== State fields (you provide these) ===================
    base_directory: str = StateField(description="Working directory for file I/O")
    # =========================================================================

    def _safe_path(self, filename: str) -> Path:
        """Resolve a file name inside base_directory, and refuse to escape it."""
        root = Path(self.base_directory).resolve()
        full = (root / filename).resolve()
        if root not in full.parents and full != root:
            raise ValueError(f"path escapes the working directory: {filename}")
        return full

    def _run(self) -> str:
        try:
            in_path = self._safe_path(self.events_file)
            out_path = self._safe_path(self.output_file)
        except ValueError as exc:
            return self.format_error(
                error="Access Denied", reason=str(exc),
                suggestion="Use a plain file name such as 'events.json'")

        if not in_path.exists():
            return self.format_error(
                error="File Not Found",
                reason="the event file does not exist",
                context=f"events_file={self.events_file}",
                suggestion="Run FetchEvents first to create it")

        events = json.loads(in_path.read_text())

        # The one line that does physics -- and it lives in the external package
        observables = dimuon_lib.compute_observables(events)

        out_path.write_text(json.dumps(observables))

        summary = dimuon_lib.summarize(observables)
        summary.update({"status": "ok", "output_file": self.output_file})
        return json.dumps(summary)


observables_tool = ComputeObservablesTool(base_directory=base_directory)
print("Created:", observables_tool.get_name())

In [ ]:
# Chain the two tools by hand: the output_file of step 1 is the events_file of step 2
result = json.loads(observables_tool.execute(events_file="events.json"))
for key, value in result.items():
    print(f"  {key:14s} {value:.2f}" if isinstance(value, float) else f"  {key:14s} {value}")

### 4.5 Look at the spectrum

The mean mass is dragged down by the continuum, but the median sits right on the
$Z$. Histogram it and the peak is unmistakable.

In [ ]:
import matplotlib.pyplot as plt

masses = json.loads((SANDBOX / "observables.json").read_text())["mass"]

fig, ax = plt.subplots(figsize=(6.5, 4))
ax.hist(masses, bins=50, range=(0, 130), histtype="stepfilled",
        color="#3b6ea5", edgecolor="#1f3f66")
ax.axvline(91.19, color="crimson", ls="--", lw=1, label=r"$m_Z = 91.19$ GeV")
ax.set_xlabel(r"dimuon invariant mass $m_{\mu\mu}$ [GeV]")
ax.set_ylabel("events / 2.6 GeV")
ax.set_title(f"Toy dimuon spectrum ({len(masses)} events)")
ax.legend()
fig.tight_layout()
plt.show()

### 4.6 Hand both tools to an agent

Everything so far ran without a language model. Now we give the two tools to an
agent and ask a question. The agent has to work out, on its own, that
it needs `FetchEvents` first and `ComputeObservables` second, and that the file
name from the first call is the input to the second.

It can only do that from your docstrings. This cell is really a test of the
prose you wrote earlier.

**This cell needs an API key.** If you do not have one, skip to Section 5 — the
tools are already working. See `heptapod/examples/orchestral/orchestral_setup_basics.ipynb` for more details.

In [ ]:
from orchestral import Agent
from orchestral.llm import GPT, Claude, Gemini, Groq
from examples.shared.tool_logger import ToolCallLogger

# Choose your provider (uncomment one)
llm = GPT()
# llm = Claude()
# llm = Gemini()
# llm = Groq()

agent = Agent(
    llm=llm,
    tools=[fetch_tool, observables_tool],
    tool_hooks=[ToolCallLogger(verbose=True, show_results=False)],
    system_prompt=(
        "You are a particle-physics analysis assistant. Use the tools provided "
        "to fetch events and compute observables. Report numbers with units."
    ),
    debug=False,
)

print(f"Agent ready with {len([fetch_tool, observables_tool])} custom tools "
      f"using {llm.__class__.__name__}")

In [ ]:
# Watch the >>> TOOL CALL lines: the agent should chain the two tools unaided
response = agent.run(
    "Pull 800 events from the DoubleMuon_RunB dataset and compute the dimuon "
    "invariant mass observables. What is the median mass, and which particle "
    "does it correspond to?"
)
print(response)

If the agent called the tools in the wrong order, passed the wrong file name, or
invented a dataset that does not exist, **the fix is almost always in a
docstring, not in the code.** Say the ordering out loud ("Call this FIRST"),
list the valid dataset names, and name the argument that carries the file
between steps. Then try again.

---

## 5. Ship it: making your tools toolbase-compatible

So far the tools only exist inside this notebook. To use them from Claude Code,
Codex, or any other agent, they have to become a **toolkit**: a directory with a
`toolkit.yaml` that [toolbase](https://github.com/alexr314/toolbase) can install
and serve over MCP.

That is a smaller step than it sounds. The tool classes do not change at all.
You are only adding a manifest.

```
cms_dimuon/                  <- the toolkit directory
├── toolkit.yaml             <- the manifest: metadata, config, bundles, tools
├── requirements.txt         <- pip dependencies
└── tools/
    ├── __init__.py
    ├── dimuon_tools.py      <- the two classes, unchanged
    └── dimuon_lib.py        <- the external physics package
```

`toolkit.yaml` answers four questions:

| Block | Question it answers |
|---|---|
| top-level metadata | what is this toolkit called? |
| `config:` | what settings must the user supply? (**these become your `StateField`s**) |
| `bundles:` | which groups can be installed separately, and what does each need from pip? |
| `tools:` | which classes should be served, and from which module? |

The `config:` $\to$ `StateField` link is the important one. Declare `db_path`
under `config:`, the user sets it with `tb config set`, and toolbase injects the
value into the `db_path` `StateField` on your tool. That is how a tool gets a
machine-specific path without anyone hard-coding it and without the model ever
seeing it.

In [ ]:
# Create the toolkit skeleton next to this notebook
TOOLKIT = Path.cwd() / "cms_dimuon"
(TOOLKIT / "tools").mkdir(parents=True, exist_ok=True)

# Vendor the "external package" into the toolkit so this example is self-contained.
# In real life you would NOT copy it -- you would list it as a pip dependency in
# requirements.txt and under the bundle's `deps:` key.
import shutil
shutil.copy(SANDBOX / "external_packages" / "dimuon_lib.py", TOOLKIT / "tools")

print(f"Toolkit skeleton at {TOOLKIT}")

In [ ]:
%%writefile cms_dimuon/tools/dimuon_tools.py
"""Two tools for the toy CMS dimuon workflow."""
import json
import sqlite3
from pathlib import Path

from orchestral.tools import BaseTool
from orchestral.tools.base.field_utils import RuntimeField, StateField

from . import dimuon_lib          # the external physics package

# NanoAOD-style field names. The Muon_* columns hold one list per event.
SCALAR_COLUMNS = ["run", "luminosityBlock", "event", "nMuon"]
MUON_COLUMNS = ["Muon_pt", "Muon_eta", "Muon_phi", "Muon_charge"]

KNOWN_DATASETS = "DoubleMuon_RunA, DoubleMuon_RunB"


def _safe_path(base_directory: str, filename: str) -> Path:
    """Resolve a file name inside base_directory, and refuse to escape it."""
    root = Path(base_directory).resolve()
    full = (root / filename).resolve()
    if root not in full.parents and full != root:
        raise ValueError(f"path escapes the working directory: {filename}")
    return full


class FetchEventsTool(BaseTool):
    """
    Fetch dimuon events from the experiment's event database and write them to a
    JSON file in the working directory, one array per field.

    Call this FIRST, before computing any observables. Available datasets are
    'DoubleMuon_RunA' and 'DoubleMuon_RunB'.

    Args:
        dataset: Name of the dataset to read.
        max_events: How many events to pull (default 500).
        output_file: File name to write, relative to the working directory.

    Returns (JSON):
        {"status": "ok", "dataset": "<name>", "n_events": <int>,
         "output_file": "<name>"}
    """

    dataset: str = RuntimeField(
        description="Dataset name, one of 'DoubleMuon_RunA' or 'DoubleMuon_RunB'")
    max_events: int = RuntimeField(
        default=500, description="Maximum number of events to fetch")
    output_file: str = RuntimeField(
        default="events.json",
        description="Output file name, relative to the working directory")

    db_path: str = StateField(description="Path to the event database file")
    base_directory: str = StateField(description="Working directory for file output")

    def _run(self) -> str:
        if not Path(self.db_path).exists():
            return self.format_error(
                error="Database Not Found",
                reason="the event database file does not exist",
                context=f"db_path={self.db_path}",
                suggestion="Set it with: tb config set cms_dimuon db_path <path>")

        try:
            out_path = _safe_path(self.base_directory, self.output_file)
        except ValueError as exc:
            return self.format_error(
                error="Access Denied", reason=str(exc),
                suggestion="Use a plain file name such as 'events.json'")

        con = sqlite3.connect(self.db_path)
        con.row_factory = sqlite3.Row
        rows = con.execute(
            f"SELECT {', '.join(SCALAR_COLUMNS + MUON_COLUMNS)} FROM Events "
            "WHERE dataset = ? ORDER BY event LIMIT ?",
            (self.dataset, self.max_events)).fetchall()
        con.close()

        if not rows:
            return self.format_error(
                error="No Events Found",
                reason=f"dataset '{self.dataset}' returned no rows",
                suggestion=f"Try one of: {KNOWN_DATASETS}")

        # Columnar layout: one array per field, the way NanoAOD stores events
        events = {col: [row[col] for row in rows] for col in SCALAR_COLUMNS}
        events.update({col: [json.loads(row[col]) for row in rows]
                       for col in MUON_COLUMNS})
        out_path.write_text(json.dumps(events))
        return json.dumps({
            "status": "ok",
            "dataset": self.dataset,
            "n_events": len(rows),
            "output_file": self.output_file,
        })


class ComputeObservablesTool(BaseTool):
    """
    Compute dimuon observables (invariant mass, pair pT) for a file of events
    produced by FetchEvents, and write them to a JSON file.

    Call this AFTER FetchEvents. The events_file argument must be the
    output_file that FetchEvents reported.

    Args:
        events_file: JSON event file written by FetchEvents, e.g. 'events.json'.
        output_file: File name for the observables, relative to the working
            directory.

    Returns (JSON):
        {"status": "ok", "output_file": "<name>", "n_events": <int>,
         "mean_mass": <GeV>, "median_mass": <GeV>,
         "min_mass": <GeV>, "max_mass": <GeV>}
    """

    events_file: str = RuntimeField(
        description="Event file to read, as reported by FetchEvents, e.g. 'events.json'")
    output_file: str = RuntimeField(
        default="observables.json",
        description="Output file name for the computed observables")

    base_directory: str = StateField(description="Working directory for file I/O")

    def _run(self) -> str:
        try:
            in_path = _safe_path(self.base_directory, self.events_file)
            out_path = _safe_path(self.base_directory, self.output_file)
        except ValueError as exc:
            return self.format_error(
                error="Access Denied", reason=str(exc),
                suggestion="Use a plain file name such as 'events.json'")

        if not in_path.exists():
            return self.format_error(
                error="File Not Found",
                reason="the event file does not exist",
                context=f"events_file={self.events_file}",
                suggestion="Run FetchEvents first to create it")

        events = json.loads(in_path.read_text())
        observables = dimuon_lib.compute_observables(events)
        out_path.write_text(json.dumps(observables))

        summary = dimuon_lib.summarize(observables)
        summary.update({"status": "ok", "output_file": self.output_file})
        return json.dumps(summary)

In [ ]:
%%writefile cms_dimuon/tools/__init__.py
"""Tools served by the cms_dimuon toolkit."""
from .dimuon_tools import FetchEventsTool, ComputeObservablesTool

__all__ = ["FetchEventsTool", "ComputeObservablesTool"]

In [ ]:
%%writefile cms_dimuon/toolkit.yaml
name: cms_dimuon
version: 0.1.0
description: Toy CMS dimuon workflow -- fetch events from a database, compute observables.
author: Your Name
email: you@example.edu
license: GPL-3.0
category: hep
python_version: "3.12"

# Settings the user supplies. Each one is injected into the matching
# StateField on any tool that declares it.
config:
  - name: db_path
    description: Path to the dimuon event database (a SQLite file).
    type: path
    required: true
  - name: base_directory
    description: Working directory the tools read and write files in.
    type: path
    required: true
    default: ${CWD}

# Installable groups. `deps:` are pip requirements for the group; use `{}` for
# a pure-python bundle with none. In a real toolkit dimuon_lib would be listed
# here too, instead of being copied into tools/.
bundles:
  dimuon:
    deps: [numpy>=2.0]

# The classes to serve. `module` is the dotted import path from the toolkit
# root; `name` is the class name.
tools:
  - module: tools.dimuon_tools
    name: FetchEventsTool
    description: Fetch dimuon events from the event database into a JSON file.
    bundle: dimuon
  - module: tools.dimuon_tools
    name: ComputeObservablesTool
    description: Compute dimuon invariant mass and pair pT for a fetched event file.
    bundle: dimuon

In [ ]:
%%writefile cms_dimuon/requirements.txt
# Python dependencies for cms_dimuon
orchestral-ai>=1.0.0
numpy>=2.0

In [ ]:
%%writefile cms_dimuon/README.md
# cms_dimuon

A toy toolkit built in `building_your_first_tool.ipynb`.

| Tool | What it does |
|---|---|
| `FetchEvents` | Pull dimuon events from the event database into a JSON file |
| `ComputeObservables` | Compute invariant mass and pair pT for a fetched event file |

## Install

```bash
tb install -e ./cms_dimuon
tb config set cms_dimuon db_path "$(pwd)/sandbox/cms_dimuon.db"
tb config set cms_dimuon base_directory "$(pwd)/sandbox"
tb activate cms_dimuon
tb connect claude-code
```

Datasets available in the toy database: `DoubleMuon_RunA`, `DoubleMuon_RunB`.

### 5.1 Validate

`tb validate` parses the manifest, imports every registered module, and reports
how many tools it found. Run it before anything else. A mistyped module path or
an undeclared bundle shows up here in a second, rather than as a confusing
failure at serve time.

**You will not want to write that `tools:` list by hand forever.** Doing it once
is worth it, because it is how you learn what the four blocks are. For a toolkit
with twenty tools, `tb ingest` writes the list for you:

```bash
tb ingest --dry-run     # print what it finds, write nothing
tb ingest               # scaffold a toolkit.yaml, or merge new tools into one
```

It discovers tools by static analysis, looking for `@define_tool` functions and
`BaseTool` subclasses, and never imports your code. Run it again after adding a
tool and it merges rather than overwrites: existing entries stay byte for byte,
including your descriptions, bundles, ordering, and comments, and only the new
tools get appended.

The conventions you buy into are worth knowing before you rely on it:

- Your tools have to be findable by that scan, so they must be `@define_tool`
  functions or `BaseTool` subclasses reachable by an import path under the
  toolkit root.
- Metadata arrives as placeholders (`name: TODO_set_toolkit_name`).
- Each `description:` is lifted from the top of the docstring and can truncate
  mid sentence, so read them before publishing.
- It writes no `config:` or `bundles:` blocks, and merged-in tools arrive with
  no `bundle:`. Your `StateField`s stay unwired until you add the matching
  `config:` entries yourself (Section 5.3).

In [ ]:
try:
    result = subprocess.run(["tb", "validate"], cwd=TOOLKIT,
                            capture_output=True, text=True)
    print(result.stdout or result.stderr)
except FileNotFoundError:
    print("toolbase is not installed. Install it with:  pip install toolbase")

### 5.2 Install, configure, connect

Four commands in a terminal, from the `examples/primer/` directory. (They are
shown here rather than run, because they build an isolated environment and edit
your agent's configuration.)

```bash
# 1. Install. -e links to your source, so edits take effect on the next serve.
tb install -e ./cms_dimuon

# 2. Tell the toolkit where things live (these fill in the StateFields).
tb config set cms_dimuon db_path "$(pwd)/sandbox/cms_dimuon.db"
tb config set cms_dimuon base_directory "$(pwd)/sandbox"

# 3. Expose the tools to the agent.
tb activate cms_dimuon

# 4. Wire up a harness and start it.
tb connect claude-code       # or: codex, opencode, orchestral
claude                       # or: codex, opencode
```

Check what the agent will actually see before launching it:

```bash
tb list -v          # every installed tool, with a check mark on the active ones
tb serve --dry-run  # what the active profile would serve
```

Inside Claude Code, `/mcp` should list a `toolbase` server carrying
`cms_dimuon__FetchEvents` and `cms_dimuon__ComputeObservables`. In OpenCode the
same check is spelled `/mcps`. Then ask, in the chat, the same question you
asked in Section 4.6:

> Pull 800 events from DoubleMuon_RunB and compute the dimuon invariant mass
> observables. What is the median mass?

Same tools, same docstrings, no Python on your side at all.

### 5.3 How the pieces line up

| In your tool class | In `toolkit.yaml` | Who supplies the value |
|---|---|---|
| `RuntimeField` | — | the model, on every call |
| `StateField` | a `config:` entry of the same name | the user, via `tb config set` |
| the class docstring | `description:` (a one-line summary) | you |
| an `import` of a pip package | the bundle's `deps:` | toolbase, at install time |

Two habits worth forming now: **a `StateField` with no `config:` entry never
gets a value**, and **a bundle whose `deps:` are incomplete installs cleanly and
then fails on import**. `tb validate` catches the second one.

### 5.4 When a docstring isn't enough: skills

A docstring documents **one tool**: its arguments, what it returns, when to call
it. A lot of what an agent needs to know does not fit that shape, because it is
not about any single tool call:

- **How the tools fit together.** `FetchEvents` runs before
  `ComputeObservables`, always. If the schema tag does not match (Section 4),
  re-fetch rather than hand-editing the file.
- **Conventions of the setup.** Which config keys have to be set before anything
  works, how the working directory is laid out, what to name outputs so the next
  step can find them.
- **Domain knowledge the model will not infer.** `DoubleMuon_RunA` is the large
  dataset, so iterate on `RunB` while the selection is still moving. A mean mass
  well below the median means the continuum is pulling it, not that the
  reconstruction is broken. Only opposite-sign pairs are $Z$ candidates.
- **What an error actually means.** The cryptic message a tool passes through
  from the software underneath it, and the fix that usually follows.
- **Templates worth copying.** A run card, a config fragment, a worked example
  that is easier to adapt than to write from scratch.

You can push a little of this into docstrings, which is what "Call this FIRST"
was doing, but it does not scale, and it has to be repeated in every tool the
advice touches.

A **skill** is where that knowledge goes: one markdown file, written for the
agent, covering what is predictable but not obvious. It sits next to your tools,
and toolbase finds it on its own, so nothing goes in `toolkit.yaml`.

```
cms_dimuon/
├── toolkit.yaml
├── tools/
└── skills/
    └── dimuon/
        ├── SKILL.md
        └── references/     # optional: templates, error tables, longer notes
            └── pitfalls.md
```

```markdown
---
name: dimuon
bundle: dimuon
description: Fetching dimuon events and computing observables. Use whenever the
  user asks for a dimuon spectrum, an invariant mass, or events from a
  DoubleMuon dataset.
---

# Dimuon analysis

Always call FetchEvents before ComputeObservables. If ComputeObservables
reports a schema mismatch, re-run FetchEvents rather than editing the file...
```

`name` and `description` are required; `bundle` is optional and ties the skill to
that bundle, so it stays hidden when those tools are not being served. The
`description` is what the agent reads when deciding whether to load the skill, so
put the trigger conditions in it, not a summary.

`tb install` and `tb connect` copy it to `~/.claude/skills/cms_dimuon__dimuon/`,
where Claude Code loads it automatically when the conversation looks relevant and
also offers it as `/cms_dimuon__dimuon`. Codex and OpenCode have no automatic
skill loading, so there it is the slash command only. Requires toolbase $\geq$ 0.11.

| Goes in the docstring | Goes in a skill |
|---|---|
| what one tool does, its arguments, what it returns | ordering across tools, setup conventions, domain knowledge, error decoding, templates |

**When to write one:** the second time you explain the same thing to the agent.
HEPTAPOD currently ships two skills, [`skills/mg5/`](../../skills/mg5/) and
[`skills/feynrules/`](../../skills/feynrules/), and between them they carry every
kind listed above: which skill to read first, physics conventions such as
declaring a mass and width exactly once, a decoder for MadGraph's less helpful
errors, and `.fr` and `.mg5` templates under `references/`. All of it exists
because the same mistakes kept recurring.

### 5.5 Adding a tool to HEPTAPOD itself

HEPTAPOD is one such toolkit, and its
[`toolkit.yaml`](../../toolkit.yaml) has exactly the blocks above, just longer.
Contributing a tool means dropping a class into `tools/<area>/`, adding one entry
under `tools:`, and running `tb validate`. See
[CONTRIBUTING.md](../../CONTRIBUTING.md) for the full checklist.

---

## 6. Checklist and common mistakes

Before you hand a tool to an agent:

- [ ] The **docstring** says what the tool does, when to call it, what each
      argument means, and what comes back.
- [ ] Ordering constraints are **stated in words** ("Call this FIRST", "requires
      the output of FetchEvents").
- [ ] Every argument the model must guess has its **valid values listed**.
- [ ] `_run()` returns a **string**, never a dict and never `None`.
- [ ] Every failure returns `format_error()` with a **`suggestion`**.
- [ ] Paths from the model are validated (`_safe_path`) before use.
- [ ] Anything the model should not choose is a **`StateField`**.
- [ ] Large data moves **through files**; the return value is a summary.
- [ ] Files passed between tools carry a **schema tag**, and the reader checks it.
- [ ] You ran it by hand with `.execute()`, including the error paths.

The mistakes that cost the most time:

| Mistake | What you see | Fix |
|---|---|---|
| Thin docstring | agent picks the wrong tool, or invents arguments | write the manual, not a label |
| Returning a `dict` | silent stringification, confused agent | `json.dumps(...)` |
| Raising an exception | agent sees a generic failure and gives up | catch it, `format_error(...)` |
| Returning all the data | context window fills, cost explodes | return a path + summary |
| Physics inside the tool | untestable, unreusable | put it in a library, wrap the library |
| Everything in one tool | agent cannot re-run one step | one tool, one job |
| `StateField` without a `config:` entry | value arrives as `None` | declare it in `toolkit.yaml` |

---

## 7. Exercises

In rough order of difficulty. Each one is a potential real pattern that might be useful IRL.

1. **A third observable.** Add $\Delta\phi$ between the two muons to
   `dimuon_lib.compute_observables()` — one more line of `numpy`, no loop.
   Note that `ComputeObservablesTool` does not change at all; that is the payoff
   of keeping physics in the library.

2. **A discovery tool.** Write `ListDatasetsTool`, which returns the dataset
   names and event counts in the database. Then remove the dataset names from
   `FetchEventsTool`'s docstring and see whether the agent finds them on its own.

3. **A cut.** Add a `min_pt` runtime field to `FetchEventsTool` that drops events
   where either muon falls below it. Ask the agent to compare the spectrum with
   and without a 20 GeV cut.

4. **A histogram tool.** Write `MakeHistogramTool`: it reads an observables file
   and returns bin edges and counts as JSON. This is closer to what a `coffea`
   processor actually produces, and it is better agent design than returning the
   observables themselves. An agent can read forty bin counts, but not a hundred
   thousand masses. Have it save a PNG too, then ask the agent for the spectrum
   and let it choose the binning.

5. **A second data source.** Point `FetchEventsTool` at a CSV file instead of
   SQLite by changing only `_run()`. The docstring, the fields, and the agent's
   view of the tool should all be unchanged.

6. **Variable muon multiplicity.** Give some events three or four muons in the
   database, so the `Muon_*` lists stop being the same length. `FetchEvents`
   barely changes; the work lands in `dimuon_lib`, which now has to *choose* two
   muons per event (highest $p_T$, opposite sign) before it can compute a mass.
   This ragged case is exactly what `awkward` exists to handle in a real
   analysis.

7. **Ship it.** Add your new tools to `cms_dimuon/toolkit.yaml`, re-run
   `tb validate`, and drive them from your favorite harness (Claude Code, Codex, opencode, etc).

8. **The real thing.** Replace the toy database with real events: open a NanoAOD
   file with `uproot` inside `FetchEvents`, and compute the observables with
   `coffea`/`awkward` inside `dimuon_lib`. The tool boundaries, the docstrings,
   and the agent's view of the workflow do not change, only what happens inside
   `_run()`.

---

## 8. Where to go next

- [`s1_lq_rr_tutorial.ipynb`](../sim/s1_lq_rr/s1_lq_rr_tutorial.ipynb) — a full
  BSM workflow: FeynRules $\to$ MadGraph $\to$ Pythia $\to$ analysis
- [CONTRIBUTING.md](../../CONTRIBUTING.md) — the checklist for contributing a
  tool to HEPTAPOD
- [tools/](../../tools/) — every shipped tool, as worked examples. Start with
  [`tools/units/`](../../tools/units/); it is the smallest.

In [ ]:
# Cleanup (optional): remove everything this notebook created
# import shutil
# shutil.rmtree(SANDBOX, ignore_errors=True)
# shutil.rmtree(TOOLKIT, ignore_errors=True)
# print("Removed sandbox/ and cms_dimuon/")

print("Done.")